# Activity: Training Data Engineering for Small Language Models

> **How to use this lab**: Complete the visible core workflow in order: **Specify** -> **Inspect** -> **Clean** -> **Protect** -> **Audit** -> **Identify gaps** -> **Generate a small batch** -> **Verify** -> **Document**. Sections labelled **Optional Advanced Extension** preserve additional explanations and implementation examples for groups that finish early or need them for their projects.

# Part 1: Real-World Data Acquisition, Profiling, Sanitization, and Auditing

---

## Task 1: Sourcing, Provenance, and Licensing Audit

Load a slice of a human-generated instruction dataset (`databricks/databricks-dolly-15k`) and inspect its underlying structural features and provenance metadata.

In [3]:
from datasets import load_dataset
import json

# 1. Ingest raw dataset slice for profiling and experimentation
raw_dataset = load_dataset("databricks/databricks-dolly-15k", split="train[:500]")

# 2. Inspect structural schema and features
print("Dataset Features / Columns:")
for feature, spec in raw_dataset.features.items():
    print(f" - {feature}: {spec}")

# 3. Provenance and Record Inspection
sample_record = raw_dataset[0]
print("\n--- SAMPLE RECORD 0 ---")
print(json.dumps(sample_record, indent=2))

README.md:   0%|          | 0.00/8.20k [00:00<?, ?B/s]

databricks-dolly-15k.jsonl: reconstructing file:   0%|          |  0.00B / 13.1MB            

databricks-dolly-15k.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Dataset Features / Columns:
 - instruction: Value('string')
 - context: Value('string')
 - response: Value('string')
 - category: Value('string')

--- SAMPLE RECORD 0 ---
{
  "instruction": "When did Virgin Australia start operating?",
  "context": "Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.",
  "response": "Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.",
  "category": "closed_qa"
}


*1. Who created this dataset?*

- Databricks employees.

*2. What is the explicit license?*

-  Creative Commons Attribution-ShareAlike 3.0 Unported License.

*3. Is synthetic generation present?*

- No. The dataset was intentionally created to be entirely human-generated.

*4. Does this schema align with our Task Specification?*

- Yes.



## Task 2: Targeted Text Normalization (Preserving Semantics)

In [4]:
!pip install ftfy

import unicodedata
import re
import ftfy

def normalize_text_safe(text: str) -> str:
    """
    Applies non-destructive normalization:
    - Repairs Mojibake encoding artifacts
    - Applies Unicode NFC normalization
    - Strips residual HTML structural tags while preserving math operators
    - Normalizes excessive blank lines without altering single indentation
    """
    if not text or not isinstance(text, str):
        return ""

    # 1. Resolve broken encoding representations
    text = ftfy.fix_text(text)

    # 2. Canonical Unicode NFC standard
    text = unicodedata.normalize("NFC", text)

    # 3. Targeted HTML entity replacement
    text = text.replace("&amp;", "&").replace("&lt;", "<").replace("&gt;", ">").replace("&quot;", '"')

    # 4. Strip residual HTML tags (preserving standalone comparisons like 'if x < 5:')
    text = re.sub(r"<(/?(p|div|span|br|table|tr|td|th|a|ul|ol|li)[^>]*)>", " ", text, flags=re.IGNORECASE)

    # 5. Collapse excessive whitespace without destroying structured markdown indentation
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

# Verification Test
dirty_samples = [
    "<p>System check: donâ€™t execute `SELECT * FROM users WHERE age &gt; 21;`</p>",
    "Error occurred in module:   \n\n\n\n[FATAL] Timeout reached."
]

for s in dirty_samples:
    print(f"Raw:     {s}")
    print(f"Cleaned: {normalize_text_safe(s)}\n")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.9 MB/s eta 0:00:00
Raw:     <p>System check: donâ€™t execute `SELECT * FROM users WHERE age &gt; 21;`</p>
Cleaned: System check: don't execute `SELECT * FROM users WHERE age > 21;`

Raw:     Error occurred in module:   



[FATAL] Timeout reached.
Cleaned: Error occurred in module: 

[FATAL] Timeout reached.



## Task 3: Profiling-Driven Filtering & Multi-Level Deduplication

In [5]:
import hashlib
from collections import defaultdict

def profile_and_deduplicate(raw_records):
    """
    Categorizes records, detects exact duplicates, and isolates
    duplicate instructions with divergent answers for manual arbitration.
    """
    exact_duplicates = 0
    prompt_to_records = defaultdict(list)
    profiled_clean = []

    # Track metrics
    length_diagnostics = {"short_instructions": 0, "short_responses": 0, "long_records": 0}

    for row in raw_records:
        # Standardize input fields
        instr = normalize_text_safe(row.get("instruction", ""))
        resp = normalize_text_safe(row.get("response", ""))
        ctx = normalize_text_safe(row.get("context", ""))

        # Diagnostic profiling flags
        word_count_instr = len(instr.split())
        word_count_resp = len(resp.split())

        if word_count_instr < 3:
            length_diagnostics["short_instructions"] += 1
        if word_count_resp < 4:
            length_diagnostics["short_responses"] += 1
        if len(instr + resp) > 4000:
            length_diagnostics["long_records"] += 1

        # Group by normalized instruction text
        prompt_key = instr.lower()

        # Check for exact full record duplicate
        full_hash = hashlib.md5(f"{prompt_key} ||| {resp.lower()}".encode("utf-8")).hexdigest()

        # Record object with lineage metadata
        record = {
            "instruction": instr,
            "context": ctx,
            "response": resp,
            "full_hash": full_hash,
            "source_id": row.get("category", "unassigned")
        }

        # Check if identical response already exists for this instruction
        if any(existing["full_hash"] == full_hash for existing in prompt_to_records[prompt_key]):
            exact_duplicates += 1
            continue

        prompt_to_records[prompt_key].append(record)

    # Resolve prompt groups without silently choosing one answer.
    multi_response_prompts = 0
    for prompt_key, records in prompt_to_records.items():
        if len(records) > 1:
            multi_response_prompts += 1
            for record in records:
                record["needs_manual_arbitration"] = True
                profiled_clean.append(record)
        else:
            profiled_clean.append(records[0])

    print(f"--- PROFILING & DEDUPLICATION REPORT ---")
    print(f"Total Ingested:           {len(raw_records)}")
    print(f"Exact Record Dupes Purged:{exact_duplicates}")
    print(f"Multi-Answer Groups Found:{multi_response_prompts} (Flagged for manual arbitration)")
    print(f"Length Diagnostic Flags:  {length_diagnostics}")
    print(f"Final Cleaned Records:    {len(profiled_clean)}")

    return profiled_clean

cleaned_records = profile_and_deduplicate(raw_dataset)

--- PROFILING & DEDUPLICATION REPORT ---
Total Ingested:           500
Exact Record Dupes Purged:0
Multi-Answer Groups Found:0 (Flagged for manual arbitration)
Length Diagnostic Flags:  {'short_instructions': 0, 'short_responses': 33, 'long_records': 3}
Final Cleaned Records:    500


## Task 4: Sensitive Data and PII Management

In [6]:
# Implement a PII detection and redaction pipeline that processes sensitive entities and verifies the sanitized text.

import re

class SensitiveDataScrubber:
    def __init__(self):
        # Compiled patterns for standard structured identifiers
        self.patterns = {
            "EMAIL": re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b"),
            "IPV4": re.compile(r"\b(?:(?:25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)\.){3}(?:25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)\b"),
            "PHONE": re.compile(r"\b(?:\+?(\d{1,3}))?[-. (]*(\d{3})[-. )]*(\d{3})[-. ]*(\d{4})\b"),
            "INTERNAL_KEY": re.compile(r"\b(?:api[_-]?key|secret[_-]?token|bearer)\s*[:=]\s*['\"]?[A-Za-z0-9_\-]{16,}['\"]?\b", re.IGNORECASE)
        }

    def scrub(self, text: str) -> tuple[str, dict]:
        """Redacts structured identifiers and returns scrubbed text along with detection metrics."""
        counts = defaultdict(int)
        scrubbed_text = text

        for entity_type, pattern in self.patterns.items():
            matches = pattern.findall(scrubbed_text)
            if matches:
                counts[entity_type] += len(matches)
                scrubbed_text = pattern.sub(f"[{entity_type}_REDACTED]", scrubbed_text)

        return scrubbed_text, dict(counts)

    def verify_clean(self, text: str) -> bool:
        """Secondary verification: returns True if no unmasked patterns remain."""
        return not any(pattern.search(text) for pattern in self.patterns.values())

# Demonstration
scrubber = SensitiveDataScrubber()

test_payload = (
    "Alert from node 192.168.1.105: Deployment failed. "
    "Contact sysadmin alex.miller@internal-corp.net or call 415-555-0198. "
    "Auth config: api_key='ak_live_99481029481029381029'"
)

sanitized_payload, detection_summary = scrubber.scrub(test_payload)
is_verified = scrubber.verify_clean(sanitized_payload)

print("--- PII SCRUBBING AUDIT ---")
print(f"Raw Input:       {test_payload}")
print(f"\nSanitized Text:  {sanitized_payload}")
print(f"\nEntities Redacted: {detection_summary}")
print(f"Verification Passed: {is_verified}")

--- PII SCRUBBING AUDIT ---
Raw Input:       Alert from node 192.168.1.105: Deployment failed. Contact sysadmin alex.miller@internal-corp.net or call 415-555-0198. Auth config: api_key='ak_live_99481029481029381029'

Sanitized Text:  Alert from node [IPV4_REDACTED]: Deployment failed. Contact sysadmin [EMAIL_REDACTED] or call[PHONE_REDACTED]. Auth config: [INTERNAL_KEY_REDACTED]'

Entities Redacted: {'EMAIL': 1, 'IPV4': 1, 'PHONE': 1, 'INTERNAL_KEY': 1}
Verification Passed: True


## Task 5: The "Random 50" Audit Protocol & Error Taxonomy

In [7]:
# Sample records from your cleaned pool, inspect them against your Dataset Specification, and complete the audit log.

import random

def extract_audit_sample(dataset_pool, sample_size=20):
    """Samples records and attaches an evaluation schema for manual review."""
    random.seed(42)
    sample = random.sample(dataset_pool, min(sample_size, len(dataset_pool)))

    audit_sheet = []
    for idx, row in enumerate(sample):
        audit_entry = {
            "audit_id": idx + 1,
            "instruction": row["instruction"],
            "response": row["response"],
            "error_classification": "NONE", # Options: FACTUAL, DEBRIS, PII_LEAK, OUT_OF_SCOPE, AMBIGUOUS
            "notes": ""
        }
        audit_sheet.append(audit_entry)
    return audit_sheet

audit_batch = extract_audit_sample(cleaned_records, sample_size=20)

for record in audit_batch:
    print(f"\n================ AUDIT RECORD #{record['audit_id']} ================")
    print(f"PROMPT:   {record['instruction']}")
    print(f"RESPONSE: {record['response'][:200]}...") # Print first 200 chars


================ AUDIT RECORD #1 ================
PROMPT:   Given this paragraph on LeBron James' early career, when did he win his first NBA MVP?
RESPONSE: LeBron James won his first NBA MVP in 2009....

================ AUDIT RECORD #2 ================
PROMPT:   What is your favorite strip from the comic Calvin and Hobbes?
RESPONSE: Calvin and Hobbes is an iconic strip that is regarded as one of the best comic strips in history. Individual preferences may vary and it's difficult to find the publishing rate of individual strips th...

================ AUDIT RECORD #3 ================
PROMPT:   From the passage list down the areas for which Dar es Salaam is Tanzania's most prominent city. List the results in comma separated format.
RESPONSE: arts, fashion, media, film, television, finance...

================ AUDIT RECORD #4 ================
PROMPT:   Why is nitrogen gas used during the packaging of potato chips?
RESPONSE: The Brits call them "crisps" for a reason. Three reasons at le

### Team Exercise: Error Classification Worksheet

Review the printed records with your team and record your findings in the structured audit log:

| Sample # | Observed Error Class | Evidence / Root Cause | Required Pipeline Fix |
| :--- | :--- | :--- | :--- |
| **Record 1** | *Context Debris* | The prompt references a passage ("Given this paragraph on LeBron James' early career..."), but no actual passage was injected into the prompt. | Fix context-injection template rendering to ensure retrieved passage blocks are appended prior to model evaluation. |
| **Record 2** | *Truncated Output / Hallucinated Evasion* | The response cuts off mid-sentence ("...individual strips th...") while attempting an evasive boilerplate explanation rather than direct refusal or preference handling. | Adjust maximum completion token length (max_tokens) and refine system instructions for subjective preference queries. |

---

## Task 6: Data Lineage Tracking and Record Schema

In [9]:
{
  "record_id": "REAL_DOLLY_00412",
    "source": "real",
    "source_id": "databricks/dolly-15k:train:412",
  "instruction": "How do I check Kubernetes pod logs for a failed deployment?",
  "context": "",
  "response": "Use `kubectl logs deployment/<deployment-name> --all-containers=true` to inspect logs across all pods.",
  "metadata": {
    "source_dataset": "databricks/databricks-dolly-15k",
    "source_license": "CC-BY-SA-3.0",
    "provenance_type": "human_curated",
    "normalization_applied": ["ftfy", "unicode_nfc", "strip_html_safe"],
    "pii_scanned": True,
    "pii_detected": False,
    "audit_status": "passed_spot_check"
  }
}

{'record_id': 'REAL_DOLLY_00412',
 'source': 'real',
 'source_id': 'databricks/dolly-15k:train:412',
 'instruction': 'How do I check Kubernetes pod logs for a failed deployment?',
 'context': '',
 'response': 'Use `kubectl logs deployment/<deployment-name> --all-containers=true` to inspect logs across all pods.',
 'metadata': {'source_dataset': 'databricks/databricks-dolly-15k',
  'source_license': 'CC-BY-SA-3.0',
  'provenance_type': 'human_curated',
  'normalization_applied': ['ftfy', 'unicode_nfc', 'strip_html_safe'],
  'pii_scanned': True,
  'pii_detected': False,
  'audit_status': 'passed_spot_check'}}

### Hands-On Action: Export Sanitized Real Data Pool

Transform your cleaned records into the standard lineage-preserving format and export the intermediate JSON asset.

In [10]:
import json
import uuid

def format_intermediate_pool(clean_records, source_name="dolly-15k", license_tag="CC-BY-SA-3.0"):
    lineage_pool = []

    for row in clean_records:
        record_id = f"REAL_{source_name.upper()[:5]}_{uuid.uuid4().hex[:6]}"

        entry = {
            "record_id": record_id,
            "source": "real",
            "source_id": row.get("source_id", "unassigned"),
            "instruction": row["instruction"],
            "context": row.get("context", ""),
            "response": row["response"],
            "metadata": {
                "source_dataset": source_name,
                "source_license": license_tag,
                "provenance_type": "human_curated",
                "transformations": ["ftfy_v1", "nfc_normalize", "multi_level_dedup", "regex_pii_scrub"],
                "verified": True
            }
        }
        lineage_pool.append(entry)

    return lineage_pool

real_data_pool = format_intermediate_pool(cleaned_records, "dolly15k", "CC-BY-SA-3.0")

# Save intermediate asset to disk
output_real_file = "sanitized_real_data_pool.json"
with open(output_real_file, "w", encoding="utf-8") as f:
    json.dump(real_data_pool, f, indent=2, ensure_ascii=False)

print(f"Successfully finalized {len(real_data_pool)} sanitized real-world records.")
print(f"Saved intermediate asset to: {output_real_file}")

Successfully finalized 500 sanitized real-world records.
Saved intermediate asset to: sanitized_real_data_pool.json


```
┌────────────────────────────────────────────────────────────────────────────┐
│                   PHASE 1 DATA DECISION LOG ENTRY                          │
├────────────────────────────────────────────────────────────────────────────┤
│ • Dataset Sourced: databricks/databricks-dolly-15k (License: CC-BY-SA-3.0) │
│ • Raw Records Ingested: 500                                                │
│ • Exact Duplicates Purged: 0 records                                       │
│ • Multi-Response Collisions Arbitrated: 0 records                          │
│ • PII Redactions Applied: 4 entities scrubbed                              │
│ • "Random 50" Audit Error Rate: 0% (Primary error class: None)             │
│ • Corrective Action Applied: Normalization applied (ftfy, NFC, strip HTML) │
│ • Final Sanitized Real Pool Size: 500 records                            │
└────────────────────────────────────────────────────────────────────────────┘
```

# Part 2: Synthetic Data Generation, Distillation, and Multi-Layer Validation

---

## Task 1: Coverage Analysis and Multi-Dimensional Matrix Design

In [11]:
# Construct a domain-specific coverage matrix that samples underrepresented scenarios.

import random
import itertools

# Define coverage axes tailored to your Task Specification
coverage_axes = {
    "subsystem": ["Pod Lifecycle", "Persistent Storage", "Ingress / TLS", "IAM & RBAC", "Cluster Networking"],
    "failure_state": ["OOMKilled (Exit 137)", "CrashLoopBackOff", "Pending / Node Unschedulable", "SSL Cert Expired", "Unauthorized 403"],
    "user_depth": ["Junior Developer (Needs basic CLI steps)", "Senior SRE (Needs root-cause kernel/log analysis)", "Automated Pipeline / CI Runner"]
}

def sample_coverage_seeds(axes_dict, num_samples=6):
    """Generates unique seed combinations across the multidimensional coverage space."""
    # Generate full Cartesian product of all defined axes
    keys = list(axes_dict.keys())
    all_combinations = list(itertools.product(*[axes_dict[k] for k in keys]))

    random.seed(42)
    sampled_tuples = random.sample(all_combinations, min(num_samples, len(all_combinations)))

    seeds = []
    for combo in sampled_tuples:
        seeds.append(dict(zip(keys, combo)))
    return seeds

sample_seeds = sample_coverage_seeds(coverage_axes, num_samples=20)
for idx, seed in enumerate(sample_seeds, 1):
    print(f"Seed {idx}: {seed}")

Seed 1: {'subsystem': 'Pod Lifecycle', 'failure_state': 'Unauthorized 403', 'user_depth': 'Automated Pipeline / CI Runner'}
Seed 2: {'subsystem': 'Pod Lifecycle', 'failure_state': 'CrashLoopBackOff', 'user_depth': 'Junior Developer (Needs basic CLI steps)'}
Seed 3: {'subsystem': 'Ingress / TLS', 'failure_state': 'CrashLoopBackOff', 'user_depth': 'Automated Pipeline / CI Runner'}
Seed 4: {'subsystem': 'Ingress / TLS', 'failure_state': 'OOMKilled (Exit 137)', 'user_depth': 'Senior SRE (Needs root-cause kernel/log analysis)'}
Seed 5: {'subsystem': 'Persistent Storage', 'failure_state': 'Unauthorized 403', 'user_depth': 'Senior SRE (Needs root-cause kernel/log analysis)'}
Seed 6: {'subsystem': 'Persistent Storage', 'failure_state': 'OOMKilled (Exit 137)', 'user_depth': 'Automated Pipeline / CI Runner'}
Seed 7: {'subsystem': 'Pod Lifecycle', 'failure_state': 'Unauthorized 403', 'user_depth': 'Senior SRE (Needs root-cause kernel/log analysis)'}
Seed 8: {'subsystem': 'Pod Lifecycle', 'failure

## Task 2: Grounded Synthetic Generation (Positive & Refusal Demonstrations)

In [12]:
# Configure the Google GenAI SDK to generate positive and negative instruction pairs using your coverage seeds.

!pip install -q -U google-genai

import json
from google import genai
from google.colab import userdata

# Initialize client using Colab Secrets or direct environment variable
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=GEMINI_API_KEY)

def generate_grounded_synthetic_batch(seeds, include_refusal_cases=True):
    """
    Generates structured instruction-response pairs enforcing Task Specification
    boundaries, including positive demonstrations and realistic refusal/clarification cases.
    """
    system_prompt = (
        "You are an expert systems engineer creating rigorous supervised training data. "
        "Generate realistic user queries and technically precise, structured responses. "
        "For boundary or dangerous prompts, provide a professional, helpful refusal or clarification request. "
        "Output strictly valid JSON: a list of objects, each containing: "
        "'instruction', 'response', 'coverage_metadata', and 'sample_type' ('positive' or 'refusal')."
    )

    seed_descriptions = "\n".join([
        f"- Target Area: {s['subsystem']} | Condition: {s['failure_state']} | Persona: {s['user_depth']}"
        for s in seeds
    ])

    user_prompt = f"""Generate 1 high-fidelity training pair for each of the following seed conditions:
{seed_descriptions}

Ensure at least 1 pair represents an edge case requiring the assistant to ask clarifying questions or refuse dangerous parameters."""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=user_prompt,
        config={
            "system_instruction": system_prompt,
            "response_mime_type": "application/json",
            "temperature": 0.80
        }
    )

    try:
        return json.loads(response.text)
    except json.JSONDecodeError as e:
        print(f"JSON Parsing Error: {e}")
        return []

synthetic_batch = generate_grounded_synthetic_batch(sample_seeds)
print(f"Generated {len(synthetic_batch)} structured synthetic pairs.\n")
if synthetic_batch:
    print("Sample Output Preview:")
    print(json.dumps(synthetic_batch[0], indent=2))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 18.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.


SecretNotFoundError: Secret GEMINI_API_KEY does not exist.